# Logic Programming in Python

We have been working through a number of examples of logic problems by hand. Such exercises can be fun and rewarding, and are certainly useful for developing our understanding. However, it is clearly not an approach that scales well and we would like to use the computer to solve logic problems.

The history of solving logic problems with the computer has a long and rich history. Noting that the earliest attempts to create AI systems were rooted in logic, and especially in **symbol manipulation**, we can look back on the history of computing and see major advances that were motivated by this problem. For example, the LISP programming language was specifically designed for symbol manipulation and introduced many innovations in programming languages including garbage collection, support for recursion, first-class functions, and proper conditional expressions not requiring a `goto`. It has had major influences on both AI and on programming language design. The *Prolog* language embodies the notions and concepts that we are studying here even more explicitly.

We are not using LISP or Prolog; we are using Python, and Python does not have any in-built symbolic manipulation or logical inference capabilities (it does include simple logical operators but these are of limited utility for our purposes. We must therefore use a library.

I have not found a library that I am particularly satisfied with. The best I have found is `kanren`, and that is what we shall use. It is not the most intuitive of packages, and the documentation has room for improvement, so we shall have to develop our understanding slowly.

Our first task is to install kanren. Before beginning, you should create a virtual environment for this module which will create a separate environment to keep it cleanly separated from other modules and thus avoid potential library version conflicts.

* Create a virtual environment in Anaconda. Please use Python version 3.13.5.
* Once you have created this environment, launch it, and install package `minikanren`.
* You will also need to install the `Jupyter` package.

We are now ready to begin solving some basic logic problem.

## Solving basic logic problems in Kanren

One of the most basic notions in kanren is the **logic variable**. This is a special datatype that has been designed to support logical inference and is part of the kenren library. The following cell imports the command for creating a logic variable and create a new logic variable called `x`.

In [1]:
from kanren import var
x = var()

This code doesn't do anything tangible, but it does set up the machinery that we need to solve problems. Here is one of the very simplest problems we can solve using logic programming. This does not yet look like a logic problem, but we should start on familiar territory:

* Find one number $x$ such that $x==5$.

In [2]:
from kanren import eq, run
run(1, x, eq(x,5))

(5,)

Note that we have already created the logic variable $x$ and we do not need to recreate it.

In Kanren, no computations are done until the `run` function is called. `run` takes three or more arguments:
* The number of results to return (0: all of them), in this example there is only one result.
* The logic variable for which we are trying to solve (`x` in this case)
* One or more **goals**. `eq` is one of kanren's built-in *goal constructor*.

Here is a more complex example that uses two logic variables and two goals. Can you understand what this code does and how it does it?

In [3]:
from kanren import vars
y, z = vars(2)
run(1, x, eq(x,z), eq(z,7))

(7,)

Another common goal constructor is `membero` which looks to see whether an item belongs to some collection. For example, the following example returns one (1) member `x` from the collection `mylist`:

In [4]:
from kanren import membero
mylist = [1,2,3,2,4,2,5]
run(1,x,membero(x,mylist))


(1,)

We can modify this easily to return two members of the collection:

In [5]:
run(2,x,membero(x,mylist))

(1, 2)

or three members of the collection:

In [6]:
run(3,x,membero(x,mylist))

(1, 2, 3)

or **all** members of the collection

In [7]:
run(0,x,membero(x,mylist))

(1, 2, 3, 2, 4, 2, 5)

Notice that `membero` returns the *set* of members of the collection and so does not contain duplicates.

## Some more goal constructors

`eq` and `membero` are two examples of kanren's inbuilt set of goal constructors. Here are two more:

* `lany`: is True if any of a specified set of sub-goals are True.
* `lall`: is True if all of the specified set of subgoals are True.
* `conde`: is True if all of some specified set of goals are True, or if all of some other specifified set of goals are True, or...

Let's see how these work. First, let's write a simple script to find numbers that occur in the first ten members of the prime number or the Fibonacci sequence. We wil first use `lany` to look for numbers that occur in either sequence, and then use `lall` to find numbers that appear in both sequences.

In [8]:
primes = [1,2,3,5,7,11,13,17,19,23]
fibonacci = [1,1,2,3,5,8,13,21,34,55]

# Find any number that is in either the Fibonacci sequence and the primes
from kanren import lany, run, membero, var
x = var()
solution = run(0,x, lany(membero(x,primes), membero(x,fibonacci)))
print(solution)

# Find any number that is in both the Fibonacci sequence and the primes
from kanren import lall
solution = run(0,x, lall(membero(x,primes), membero(x,fibonacci)))
print(solution)


(1, 1, 2, 1, 3, 2, 5, 3, 7, 5, 11, 8, 13, 13, 17, 21, 19, 34, 23, 55)
(1, 2, 3, 5, 13, 1)


Now let us use the `conde` goal constructor to find all numbers that are in either the first ten *odd* numbers and the first ten primes, or the first ten odds and the first ten Fibonacci numbers.

In [9]:
from kanren import conde
odds = [1,3,5,7,9,11,13,15,17,19]
solution = run(0, x,
               conde(
                   (membero(x,odds),membero(x,primes)),
                   (membero(x,odds),membero(x,fibonacci))
                   )
                )
print(solution)

(1, 1, 3, 3, 5, 5, 7, 13, 11, 1, 13, 17, 19)


These goal constructors are very powerful. Can you see an equivalence between them and the logic operators that we have become familar with?

Let us now see how we can use Kanren to solve natural deduction problems.

## Natural Deduction in Kanren

So far, we have used Kanren to reason about numerical types, but it is just as easy to reason about logical types. Recasting our trivial example  of *find $x$ such that x is equal to 5* in terms of logic, we can easily write:


In [10]:
from kanren import run, var, eq
P = var()
run(0, P, eq(P, True))

(True,)

Let's use this to verify some simple proofs. Let's start with a really trivial one:

$P\land Q$

$\therefore P$

Here, we have one goal: $P\land Q$. How do we express this in kanren? Which of the goal constructors we have introduced could we use for this?

In [11]:
from kanren import run, var, eq, lall
P = var()
Q = var()
goals = lall(
    eq(P, True),
    eq(Q, True)
)
run(0, P, goals)

(True,)

Does this behave as you expect?

We will sometimes find it helpful to wrap goals up in a function. For example, we could write

In [12]:
from kanren import run, var, eq, lall
P = var()
Q = var()

def land(A,B):
    return lall(
        eq(A, True),
        eq(B, True)
    )

goals = land(P,Q)
run(0, P, goals)

(True,)

 Let's try another example. Consider the proposition

$P\lor Q$

$\lnot Q$

$\therefore P$



In [13]:
from kanren import run, var, eq, lany
P = var()
Q = var()

def lor(A,B):
    return lany(
        eq(A, True),
        eq(B, True)
    )

goals = lall(
    lor(P,Q),
    eq(Q, False)
)
run(0, P, goals)

(True,)

It is interesting to see what would have happened if the goal is ambiguous:

$P\lor Q$

What can we say about $P$? How does kanren handle this?

In [14]:
goals = lall(
    lor(P,Q),
)
run(0, P, goals)

(True, ~_808)


How should we interpret this result?

Let us now verify a slightly more involved proof:

$P \land \lnot Q$

$Q \lor R$

$\therefore R$

In [15]:
from kanren import var, eq, lall, run
P = var()
Q = var()
R = var()

goals = lall(
    lall(eq(P, True), eq(Q, False)),
    lor(Q,R)
)

run(0, R, goals)

(True,)

It might have been very tempting to formulate this as:

In [16]:
goals = lall(
    land(P, not Q),
    lor(Q,R)
)
run(0, R, goals)

()


Sadly this does not work: kanren does not support the use of the Python built-in logic operators. This can make the formulation of goals somewhat challenging.

Let us now see what happens when a problem cannot be solved. Consider the following inconsistent facts (goals):

$P\land Q$

$\lnot P$

In [17]:
goals = lall(
    land(P,Q),
    eq(P, False)
)
run(0, Q, goals)

()

The solver returns the empty set. There is no state of $Q$ which satisfies all of the goals.

We conclude by showing an alternative way to approach this, which we will need when solving a problem where were are trying to prove something a more complex expression. For example, how do we prove the argument $P,\ Q,\ :\ P\land Q$? The answer is quite easy: we include the thing we are trying to prove as one of the goals, and then see if there is any combination of variables that satisfies both our rule base and the expression we want to prove.

In [18]:
goals = lall(
    eq(P,True),
    eq(Q,True),
    lall(eq(P,True),eq(Q,True))
)

run(0,[P,Q],goals)

([True, True],)